# 4A — Multi-Temperature Nested ESN–XGBoost Optimization

This notebook performs expensive, leakage-safe joint hyperparameter optimization for
unseen-material generalization using the pooled 30–60°C dataset.

For every outer LOMO fold, one complete material—including all four temperatures and
six repetitions—is untouched. Joint ESN–XGBoost Bayesian optimization uses only the
remaining materials. Outer predictions are pooled for the primary unbiased estimate.

After nested evaluation, one final all-material grouped search produces the ESN and
XGBoost configuration consumed by Notebook 4B.


# 1. Imports and experiment configuration


In [12]:
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy import linalg
from scipy.signal import savgol_filter
from scipy.stats import kurtosis, norm, skew
from sklearn.compose import TransformedTargetRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

#


In [2]:
TEMPERATURE_FOLDERS = ("30C", "40C", "50C", "60C")
OPTIMIZATION_TARGET = "eff"
RANDOM_STATE = 42
RESERVOIR_SEEDS = (42, 43, 44)

ANALYSIS_WINDOW = (0.0, 5.0)
EARLY_WINDOW = (0.0, 1.0)
MID_WINDOW = (1.0, 3.0)
LATE_WINDOW = (3.0, 5.0)

# Defaults are used only if a function is called without an explicit candidate.
ESN_RES_SIZE = 20
ESN_LEAK_RATE = 0.1
ESN_INPUT_MAGNITUDE = 1.0
ESN_SPECTRAL_RADIUS = 0.9
ESN_WASHOUT = 0

# Baseline values are overridden by every joint Bayesian candidate.
XGB_PARAMS = {
    "n_estimators": 300, "max_depth": 3, "learning_rate": 0.03,
    "subsample": 0.85, "colsample_bytree": 0.85,
    "min_child_weight": 2.0, "reg_alpha": 0.0, "reg_lambda": 1.0,
}

BAYES_N_TRIALS = 50
BAYES_N_INITIAL = 15
BAYES_ACQUISITION_CANDIDATES = 2048
BAYES_STABILITY_WEIGHT = 0.25
INNER_MATERIAL_FOLDS = 4

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in (cwd, cwd.parent) if (path / "data").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from the project root or notebooks directory.")
DATA_ROOT = PROJECT_ROOT / "data" / "02_preprocessed"
missing_folders = [name for name in TEMPERATURE_FOLDERS if not (DATA_ROOT / name).is_dir()]
if missing_folders:
    raise FileNotFoundError(f"Missing temperature data folders: {missing_folders}")

RESULTS_DIR = PROJECT_ROOT / "results" / "multi_temp_esn"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
temperature_tag = "-".join(TEMPERATURE_FOLDERS)
RESULT_STEM = f"{temperature_tag}_{OPTIMIZATION_TARGET}_v5"
PARAMETER_FILE = RESULTS_DIR / f"{RESULT_STEM}_final_parameters.json"

print("Data root:", DATA_ROOT)
print("Temperature folders:", TEMPERATURE_FOLDERS)
print("Final parameter file:", PARAMETER_FILE)


Data root: /Users/markkeanujamesexconde/Library/CloudStorage/OneDrive-Personal/Documents/Academe Files/4. Doctoral/Research 1 - Warmth Sensor/Warmth-Sensor-Object-Detection/data/02_preprocessed
Temperature folders: ('30C', '40C', '50C', '60C')
Final parameter file: /Users/markkeanujamesexconde/Library/CloudStorage/OneDrive-Personal/Documents/Academe Files/4. Doctoral/Research 1 - Warmth Sensor/Warmth-Sensor-Object-Detection/results/multi_temp_esn/30C-40C-50C-60C_eff_v5_final_parameters.json


# 2. Standard material properties and trial loading

The processed CSV values in `k`, `Mass`, `Volume`, `rho`, and `cp` are deliberately
ignored. After loading the sensor data, the notebook overwrites those fields using
the authoritative table below.

`Mass=1 kg` and `Volume=1 m³` are placeholders and can be updated later.


In [3]:
STANDARD_PROPERTIES = {
    "ps_foam":   {"k": 0.034, "Mass": 1.0, "Volume": 1.0, "rho": 25.0,   "cp": 1400.0},
    "pu_foam":   {"k": 0.043, "Mass": 1.0, "Volume": 1.0, "rho": 30.0,   "cp": 1400.0},
    "cork":      {"k": 0.043, "Mass": 1.0, "Volume": 1.0, "rho": 240.0,  "cp": 1800.0},
    "wood":      {"k": 0.150, "Mass": 1.0, "Volume": 1.0, "rho": 700.0,  "cp": 1700.0},
    "pdms":      {"k": 0.150, "Mass": 1.0, "Volume": 1.0, "rho": 970.0,  "cp": 1460.0},
    "gypsum":    {"k": 0.170, "Mass": 1.0, "Volume": 1.0, "rho": 800.0,  "cp": 1090.0},
    "cement":    {"k": 0.290, "Mass": 1.0, "Volume": 1.0, "rho": 1440.0, "cp": 750.0},
    "graphite":  {"k": 100.0, "Mass": 1.0, "Volume": 1.0, "rho": 1820.0, "cp": 710.0},
    "bismuth":   {"k": 8.1,   "Mass": 1.0, "Volume": 1.0, "rho": 9780.0, "cp": 130.0},
    "titanium":  {"k": 21.9,  "Mass": 1.0, "Volume": 1.0, "rho": 4506.0, "cp": 523.0},
    "nickel":    {"k": 90.9,  "Mass": 1.0, "Volume": 1.0, "rho": 8908.0, "cp": 461.0},
    "iron":      {"k": 80.4,  "Mass": 1.0, "Volume": 1.0, "rho": 7874.0, "cp": 449.0},
    "aluminum":  {"k": 237.0, "Mass": 1.0, "Volume": 1.0, "rho": 2700.0, "cp": 897.0},
    "copper":    {"k": 401.0, "Mass": 1.0, "Volume": 1.0, "rho": 8960.0, "cp": 385.0},
}

SAMPLE_ALIASES = {
    "ps": "ps_foam", "ps foam": "ps_foam", "ps_foam": "ps_foam",
    "pu": "pu_foam", "pu foam": "pu_foam", "pu_foam": "pu_foam",
    "cork": "cork", "cork fine": "cork", "cork_fine": "cork",
    "wood": "wood",
    "pdms": "pdms",
    "gypsum": "gypsum",
    "cement": "cement",
    "graphite": "graphite", "carbon": "graphite",
    "bi": "bismuth", "bismuth": "bismuth",
    "ti": "titanium", "titanium": "titanium",
    "ni": "nickel", "nickel": "nickel",
    "fe": "iron", "iron": "iron",
    "al": "aluminum", "aluminum": "aluminum",
    "cu": "copper", "copper": "copper",
}

standard_table = (
    pd.DataFrame.from_dict(STANDARD_PROPERTIES, orient="index")
    .rename_axis("Material")
    .reset_index()
)
display(standard_table)

required = {"Sample", "Trial", "Time", "Primary", "Secondary"}
frames = []
load_rows = []

for temperature in TEMPERATURE_FOLDERS:
    folder = DATA_ROOT / temperature
    for path in sorted(folder.glob("*.csv")):
        frame = pd.read_csv(path)
        missing = required - set(frame.columns)
        if missing:
            warnings.warn(
                f"Skipping {temperature}/{path.name}; missing {sorted(missing)}"
            )
            continue
        frame["Temperature"] = temperature
        frame["source_file"] = path.name
        frames.append(frame)
        load_rows.append({
            "Temperature": temperature,
            "file": path.name,
            "rows_loaded": len(frame),
        })

if not frames:
    raise ValueError("No valid trial files were found.")

DATA = pd.concat(frames, ignore_index=True)
DATA = DATA.replace([np.inf, -np.inf], np.nan)
for column in ("Trial", "Time", "Primary", "Secondary"):
    DATA[column] = pd.to_numeric(DATA[column], errors="coerce")
DATA = DATA.dropna(subset=list(required) + ["Temperature"]).copy()
DATA["Trial"] = DATA["Trial"].astype(int)
DATA["Temperature_C"] = pd.to_numeric(
    DATA["Temperature"].str.extract(r"(\d+(?:\.\d+)?)", expand=False),
    errors="coerce",
)
if DATA["Temperature_C"].isna().any():
    raise ValueError("A temperature-folder name could not be converted to Celsius.")

normalized_sample = (
    DATA["Sample"].astype(str).str.strip().str.lower().str.replace("_", " ")
)
DATA["Sample"] = normalized_sample.map(SAMPLE_ALIASES)
unknown_mask = DATA["Sample"].isna()
if unknown_mask.any():
    unknown = sorted(normalized_sample[unknown_mask].unique())
    raise KeyError(f"No standard-property mapping for samples: {unknown}")

# Ignore processed-file property cells and apply one standard table everywhere.
for property_name in ("k", "Mass", "Volume", "rho", "cp"):
    DATA[property_name] = DATA["Sample"].map(
        lambda sample: STANDARD_PROPERTIES[sample][property_name]
    )

# Temperature is required in the ID because material/trial numbers repeat by folder.
DATA["trial_id"] = (
    DATA["Temperature"].astype(str)
    + "__" + DATA["Sample"].astype(str)
    + "_trial_" + DATA["Trial"].astype(str)
)
DATA["eff"] = np.sqrt(DATA["k"] * DATA["rho"] * DATA["cp"])
DATA = DATA.sort_values(["Temperature", "trial_id", "Time"]).reset_index(drop=True)

summary = (
    DATA.groupby(["trial_id", "Temperature", "Sample", "Trial"], as_index=False)
    .agg(
        n_timesteps=("Time", "size"),
        k=("k", "first"),
        eff=("eff", "first"),
    )
)
temperature_summary = (
    summary.groupby("Temperature", as_index=False)
    .agg(
        trials=("trial_id", "nunique"),
        materials=("Sample", "nunique"),
        minimum_timesteps=("n_timesteps", "min"),
        maximum_timesteps=("n_timesteps", "max"),
    )
)
print(f"Rows: {len(DATA):,}")
print(f"Unique temperature-specific trials: {DATA['trial_id'].nunique()}")
display(temperature_summary)


,Material,k,Mass,Volume,rho,cp
0,ps_foam,0.034,1.0,1.0,25.0,1400.0
1,pu_foam,0.043,1.0,1.0,30.0,1400.0
2,cork,0.043,1.0,1.0,240.0,1800.0
3,wood,0.150,1.0,1.0,700.0,1700.0
4,pdms,0.150,1.0,1.0,970.0,1460.0
5,gypsum,0.170,1.0,1.0,800.0,1090.0
6,cement,0.290,1.0,1.0,1440.0,750.0
7,graphite,100.000,1.0,1.0,1820.0,710.0
8,bismuth,8.100,1.0,1.0,9780.0,130.0
9,titanium,21.900,1.0,1.0,4506.0,523.0


Rows: 70,480
Unique temperature-specific trials: 336


,Temperature,trials,materials,minimum_timesteps,maximum_timesteps
0,30C,84,14,184,227
1,40C,84,14,200,362
2,50C,84,14,183,404
3,60C,84,14,186,326


# 3. Detect contact and align every trial

The elbow is used only to establish a common time origin. Prediction uses the fixed
0–5 second response after contact. Trials are never aligned using `k` or `eff`.


In [4]:
def find_contact_time(
    trial,
    smooth_window=15,
    polyorder=2,
    threshold_frac=0.30,
    skip_samples=5,
):
    clean = (
        trial[["Time", "Primary"]]
        .apply(pd.to_numeric, errors="coerce")
        .dropna()
        .sort_values("Time")
        .drop_duplicates("Time")
        .reset_index(drop=True)
    )
    time = clean["Time"].to_numpy(float)
    signal = clean["Primary"].to_numpy(float)
    if len(signal) < skip_samples + 7 or np.any(np.diff(time) <= 0):
        raise ValueError("Insufficient or invalid time samples.")

    work_time = time[skip_samples:]
    work_signal = signal[skip_samples:]
    window = min(int(smooth_window), len(work_signal))
    if window % 2 == 0:
        window -= 1
    minimum = polyorder + 2
    if minimum % 2 == 0:
        minimum += 1
    if window < minimum:
        raise ValueError("Sequence is too short for smoothing.")

    smooth = savgol_filter(work_signal, window, polyorder, mode="interp")
    derivative = np.gradient(smooth, work_time)
    strongest = int(np.argmin(derivative))
    active = derivative < threshold_frac * derivative[strongest]
    elbow = 0
    for position in range(strongest, -1, -1):
        if not active[position]:
            elbow = position + 1
            break
    return float(work_time[elbow])


In [5]:
aligned_trials = {}
alignment_rows = []

for trial_id, trial in DATA.groupby("trial_id", sort=False):
    trial = trial.sort_values("Time").drop_duplicates("Time").copy()
    try:
        contact_time = find_contact_time(trial)
    except ValueError as exc:
        warnings.warn(f"Skipping {trial_id}: {exc}")
        continue

    trial["time_from_contact"] = trial["Time"] - contact_time
    start, end = ANALYSIS_WINDOW
    trial = trial[
        trial["time_from_contact"].between(start, end, inclusive="both")
    ].copy()
    if len(trial) < 10:
        warnings.warn(f"Skipping {trial_id}: too few post-contact samples")
        continue
    aligned_trials[trial_id] = trial.reset_index(drop=True)
    alignment_rows.append({
        "trial_id": trial_id,
        "Temperature": trial["Temperature"].iloc[0],
        "Temperature_C": float(trial["Temperature_C"].iloc[0]),
        "Sample": trial["Sample"].iloc[0],
        "Trial": int(trial["Trial"].iloc[0]),
        "contact_time": contact_time,
        "n_analysis_samples": len(trial),
    })

ALIGNMENT = pd.DataFrame(alignment_rows)
print(f"Aligned trials retained: {len(aligned_trials)}")
display(ALIGNMENT.head())


Aligned trials retained: 336


,trial_id,Temperature,Temperature_C,Sample,Trial,contact_time,n_analysis_samples
0,30C__aluminum_trial_1,30C,30.0,aluminum,1,8.40066,50
1,30C__aluminum_trial_2,30C,30.0,aluminum,2,7.40070,50
2,30C__aluminum_trial_3,30C,30.0,aluminum,3,7.70070,51
3,30C__aluminum_trial_4,30C,30.0,aluminum,4,8.70070,51
4,30C__aluminum_trial_5,30C,30.0,aluminum,5,7.20070,51


# 4. Interpretable thermal-response features


In [6]:
def _smooth(values, window=11, polyorder=2):
    values = np.asarray(values, float)
    selected = min(window, len(values))
    if selected % 2 == 0:
        selected -= 1
    minimum = polyorder + 2
    if minimum % 2 == 0:
        minimum += 1
    return (
        savgol_filter(values, selected, polyorder, mode="interp")
        if selected >= minimum else values.copy()
    )


def _slope(time, values, window):
    mask = (time >= window[0]) & (time <= window[1])
    if mask.sum() < 3:
        return np.nan
    return float(np.polyfit(time[mask], values[mask], 1)[0])


def extract_thermal_features(trial):
    time = trial["time_from_contact"].to_numpy(float)
    primary = _smooth(trial["Primary"].to_numpy(float))
    secondary = _smooth(trial["Secondary"].to_numpy(float))
    difference = primary - secondary
    result = {}

    for name, signal in {
        "primary": primary,
        "secondary": secondary,
        "difference": difference,
    }.items():
        change = signal - signal[0]
        rate = np.gradient(signal, time)
        result[f"{name}_final_change"] = float(change[-1])
        result[f"{name}_max_abs_change"] = float(np.max(np.abs(change)))
        result[f"{name}_response_auc"] = float(np.trapezoid(np.abs(change), time))
        result[f"{name}_early_slope"] = _slope(time, signal, EARLY_WINDOW)
        result[f"{name}_mid_slope"] = _slope(time, signal, MID_WINDOW)
        result[f"{name}_late_slope"] = _slope(time, signal, LATE_WINDOW)
        result[f"{name}_max_abs_rate"] = float(np.max(np.abs(rate)))
        result[f"{name}_rate_auc"] = float(np.trapezoid(np.abs(rate), time))

    # Dimensionless/cross-sensor summaries.
    primary_auc = result["primary_response_auc"]
    result["secondary_primary_auc_ratio"] = (
        result["secondary_response_auc"] / primary_auc
        if not np.isclose(primary_auc, 0) else np.nan
    )
    result["initial_sensor_difference"] = float(difference[0])
    result["final_sensor_difference"] = float(difference[-1])
    return result


thermal_rows = []
for trial_id, trial in aligned_trials.items():
    features = extract_thermal_features(trial)
    features.update({
        "trial_id": trial_id,
        "Temperature": trial["Temperature"].iloc[0],
        "Temperature_C": float(trial["Temperature_C"].iloc[0]),
        "Sample": trial["Sample"].iloc[0],
        "Trial": int(trial["Trial"].iloc[0]),
        "k": float(trial["k"].iloc[0]),
        "eff": float(trial["eff"].iloc[0]),
    })
    thermal_rows.append(features)

THERMAL_FEATURES = pd.DataFrame(thermal_rows)
print(f"Thermal features: {len(THERMAL_FEATURES.columns) - 7}")
display(THERMAL_FEATURES.head())


Thermal features: 27


,primary_final_change,primary_max_abs_change,primary_response_auc,primary_early_slope,primary_mid_slope,primary_late_slope,primary_max_abs_rate,primary_rate_auc,secondary_final_change,secondary_max_abs_change,...,secondary_primary_auc_ratio,initial_sensor_difference,final_sensor_difference,trial_id,Temperature,Temperature_C,Sample,Trial,k,eff
0,-0.000003,0.000003,0.000013,-0.000002,-3.654261e-07,-1.344399e-07,0.000003,0.000003,-0.000002,0.000002,...,0.225534,-0.000040,-0.000042,30C__aluminum_trial_1,30C,30.0,aluminum,1,237.0,23958.094665
1,-0.000003,0.000003,0.000014,-0.000002,-3.595754e-07,-1.291998e-07,0.000004,0.000003,-0.000002,0.000002,...,0.241267,-0.000040,-0.000042,30C__aluminum_trial_2,30C,30.0,aluminum,2,237.0,23958.094665
2,-0.000003,0.000003,0.000014,-0.000002,-3.389437e-07,-1.112019e-07,0.000003,0.000003,-0.000002,0.000002,...,0.253393,-0.000040,-0.000042,30C__aluminum_trial_3,30C,30.0,aluminum,3,237.0,23958.094665
3,-0.000003,0.000003,0.000013,-0.000002,-3.724766e-07,-1.181206e-07,0.000003,0.000003,-0.000002,0.000002,...,0.243614,-0.000040,-0.000042,30C__aluminum_trial_4,30C,30.0,aluminum,4,237.0,23958.094665
4,-0.000003,0.000003,0.000013,-0.000002,-3.853054e-07,-1.244971e-07,0.000003,0.000003,-0.000002,0.000002,...,0.264204,-0.000041,-0.000042,30C__aluminum_trial_5,30C,30.0,aluminum,5,237.0,23958.094665


# 5. Manual ESN reservoir and compact trajectory summaries

Each temperature-specific trial becomes one row. Every reservoir unit contributes
nine summaries: mean, standard deviation, range, net change, slope, absolute area,
time of maximum absolute activation, skewness, and excess kurtosis.


In [7]:
class ManualReservoir:
    def __init__(
        self,
        res_size=30,
        leak_rate=0.9,
        input_magnitude=1.5,
        spectral_radius=1.3,
        washout=0,
        random_state=42,
    ):
        self.res_size = int(res_size)
        self.leak_rate = float(leak_rate)
        self.input_magnitude = float(input_magnitude)
        self.spectral_radius = float(spectral_radius)
        self.washout = int(washout)
        rng = np.random.default_rng(random_state)
        self.Win = (rng.random((self.res_size, 1 + 5)) - 0.5) * self.input_magnitude
        W = rng.random((self.res_size, self.res_size)) - 0.5
        radius = np.max(np.abs(linalg.eigvals(W)))
        if not np.isfinite(radius) or np.isclose(radius, 0):
            raise ValueError("Invalid reservoir spectral radius.")
        self.W = (W / radius.real) * self.spectral_radius

    def run(self, sequence):
        sequence = np.asarray(sequence, float)
        if sequence.ndim != 2 or sequence.shape[1] != 5:
            raise ValueError("Expected sequence with five input channels.")
        if len(sequence) <= self.washout:
            raise ValueError(
                f"Sequence has {len(sequence)} samples, but washout={self.washout}. "
                "Washout must be smaller than the sequence length."
            )
        x = np.zeros((self.res_size, 1))
        states = []
        for row in sequence:
            u = row.reshape(-1, 1)
            x = (
                (1 - self.leak_rate) * x
                + self.leak_rate * np.tanh(
                    self.Win @ np.vstack((1.0, u)) + self.W @ x
                )
            )
            states.append(x[:, 0].copy())
        return np.asarray(states)[self.washout:]


def raw_esn_input(trial):
    time = trial["time_from_contact"].to_numpy(float)
    primary = _smooth(trial["Primary"].to_numpy(float))
    secondary = _smooth(trial["Secondary"].to_numpy(float))
    difference = primary - secondary
    primary_rate = np.gradient(primary, time)
    secondary_rate = np.gradient(secondary, time)
    return np.column_stack([
        primary, secondary, difference, primary_rate, secondary_rate
    ])


def summarize_reservoir_trajectories(time, states):
    """Return finite statistical/dynamical summaries for every reservoir unit."""
    time = np.asarray(time, float)
    states = np.asarray(states, float)
    if states.ndim != 2 or len(time) != len(states):
        raise ValueError("Time and reservoir states must have matching rows.")
    if len(time) < 2 or np.any(~np.isfinite(time)) or np.any(np.diff(time) <= 0):
        raise ValueError("Reservoir-state time must be finite and strictly increasing.")
    if np.any(~np.isfinite(states)):
        raise ValueError("Reservoir states contain non-finite values.")

    result = {}
    for unit in range(states.shape[1]):
        values = states[:, unit]
        maximum_absolute_index = int(np.argmax(np.abs(values)))
        standard_deviation = float(np.std(values, ddof=0))

        # Constant or nearly constant trajectories have well-defined zero shape
        # rather than scipy's otherwise undefined skewness/kurtosis warnings.
        if len(values) < 3 or np.isclose(standard_deviation, 0.0):
            skewness = 0.0
        else:
            skewness = float(skew(values, bias=False))
        if len(values) < 4 or np.isclose(standard_deviation, 0.0):
            excess_kurtosis = 0.0
        else:
            excess_kurtosis = float(kurtosis(values, fisher=True, bias=False))

        summaries = {
            "mean": float(np.mean(values)),
            "std": standard_deviation,
            "range": float(np.ptp(values)),
            "net_change": float(values[-1] - values[0]),
            "slope": float(np.polyfit(time, values, 1)[0]),
            "absolute_area": float(np.trapezoid(np.abs(values), time)),
            "time_of_max_absolute": float(time[maximum_absolute_index]),
            "skewness": skewness,
            "excess_kurtosis": excess_kurtosis,
        }
        
        for name, value in summaries.items():
            if not np.isfinite(value):
                raise ValueError(f"Non-finite {name} for reservoir unit {unit}.")
            result[f"esn_{name}_u{unit:03d}"] = value
    return result


# 6. Leakage-safe metrics and XGBoost construction


In [8]:
def regression_metrics(y_true, y_pred):
    """Metrics requiring variation in y_true; intended for pooled or mixed-target data."""
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    if len(y_true) == 0 or np.any(~np.isfinite(y_true)) or np.any(~np.isfinite(y_pred)):
        raise ValueError("Metrics require non-empty, finite targets and predictions.")
    target_range = float(np.ptp(y_true))
    if len(y_true) < 2 or target_range <= 0:
        raise ValueError(
            "R² and test-range NRMSE require at least two distinct target values. "
            "Use constant_target_fold_metrics for a single-material LOMO fold."
        )
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    return {
        "n": len(y_true),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": rmse,
        "nrmse_range": rmse / target_range,
        "r2": float(r2_score(y_true, y_pred)),
        "median_ape_pct": float(np.median(np.abs((y_true - y_pred) / y_true)) * 100),
    }


def constant_target_fold_metrics(y_true, y_pred, training_target_range):
    """Valid diagnostics for one outer LOMO fold with a constant test target."""
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    training_target_range = float(training_target_range)
    if len(y_true) == 0 or np.any(~np.isfinite(y_true)) or np.any(~np.isfinite(y_pred)):
        raise ValueError("Fold metrics require non-empty, finite values.")
    if training_target_range <= 0:
        raise ValueError("The outer-training target range must be positive.")
    residual = y_pred - y_true
    rmse = float(np.sqrt(np.mean(residual ** 2)))
    return {
        "n": len(y_true),
        "mae": float(np.mean(np.abs(residual))),
        "rmse": rmse,
        "nrmse_training_range": rmse / training_target_range,
        "mean_error_bias": float(np.mean(residual)),
        "median_ape_pct": float(np.median(np.abs(residual / y_true)) * 100),
    }


def make_regressor(xgb_params=None, random_state=RANDOM_STATE):
    params = {**XGB_PARAMS, **(xgb_params or {})}
    xgb = XGBRegressor(
        objective="reg:squarederror",
        random_state=int(random_state),
        n_jobs=-1,
        tree_method="hist",
        verbosity=0,
        **params,
    )
    base = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("xgb", xgb),
    ])
    return TransformedTargetRegressor(
        regressor=base,
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    )


# 7. Fold-local multi-temperature ESN feature generation


In [9]:
def build_esn_feature_table(train_ids, all_ids, esn_params=None, random_state=RANDOM_STATE):
    params = {
        "res_size": ESN_RES_SIZE,
        "leak_rate": ESN_LEAK_RATE,
        "input_magnitude": ESN_INPUT_MAGNITUDE,
        "spectral_radius": ESN_SPECTRAL_RADIUS,
        "washout": ESN_WASHOUT,
    }
    params.update(esn_params or {})
    scaler = StandardScaler().fit(np.vstack([
        raw_esn_input(aligned_trials[trial_id]) for trial_id in train_ids
    ]))
    reservoir = ManualReservoir(**params, random_state=random_state)
    rows = []
    for trial_id in all_ids:
        trial = aligned_trials[trial_id]
        sequence = scaler.transform(raw_esn_input(trial))
        states = reservoir.run(sequence)
        state_time = trial["time_from_contact"].to_numpy(float)[params["washout"]:]
        features = summarize_reservoir_trajectories(state_time, states)
        features.update({
            "trial_id": trial_id,
            "Temperature": trial["Temperature"].iloc[0],
            "Temperature_C": float(trial["Temperature_C"].iloc[0]),
            "Sample": trial["Sample"].iloc[0],
            "Trial": int(trial["Trial"].iloc[0]),
            "k": float(trial["k"].iloc[0]),
            "eff": float(trial["eff"].iloc[0]),
        })
        rows.append(features)
    return pd.DataFrame(rows)

# 8. Nested joint Bayesian LOMO optimization

The inner folds are grouped by material and retain every temperature belonging to a
validation material. `Temperature_C` is supplied to XGBoost as a known operating
condition; it is not used as a target. The outer held-out material never influences
scaling, feature generation, hyperparameter selection, or model fitting.


In [10]:
ESN_BAYES_SPACE = {
    "res_size": ("integer", 10, 80),
    "leak_rate": ("float", 0.05, 0.95),
    "input_magnitude": ("float", 0.25, 2.00),
    "spectral_radius": ("float", 0.30, 1.50),
    "washout": ("integer", 0, 5),
}

XGB_BAYES_SPACE = {
    "n_estimators": ("integer", 100, 600),
    "max_depth": ("integer", 1, 4),
    "learning_rate": ("log_float", 0.01, 0.15),
    "min_child_weight": ("log_float", 1.0, 20.0),
    "subsample": ("float", 0.60, 1.00),
    "colsample_bytree": ("float", 0.30, 1.00),
    "reg_alpha": ("log_float", 1e-4, 10.0),
    "reg_lambda": ("log_float", 0.10, 100.0),
}

BAYES_SPACE = {**ESN_BAYES_SPACE, **XGB_BAYES_SPACE}
ESN_BAYES_KEYS = tuple(ESN_BAYES_SPACE)
XGB_BAYES_KEYS = tuple(XGB_BAYES_SPACE)
BAYES_KEYS = tuple(BAYES_SPACE)


def _decode_bayes_point(point):
    """Map one unit-hypercube point to a valid joint ESN-XGBoost candidate."""
    params = {}
    for key, coordinate in zip(BAYES_KEYS, np.clip(point, 0.0, 1.0)):
        kind, low, high = BAYES_SPACE[key]
        coordinate = float(coordinate)
        if kind == "integer":
            value = int(round(low + coordinate * (high - low)))
        elif kind == "log_float":
            value = float(np.exp(np.log(low) + coordinate * (np.log(high) - np.log(low))))
        elif kind == "float":
            value = float(low + coordinate * (high - low))
        else:
            raise ValueError(f"Unknown Bayesian parameter type: {kind}")
        params[key] = value
    return params


def _split_joint_parameters(params):
    missing = set(BAYES_KEYS) - set(params)
    if missing:
        raise KeyError(f"Joint candidate is missing parameters: {sorted(missing)}")
    esn_params = {key: params[key] for key in ESN_BAYES_KEYS}
    xgb_params = {key: params[key] for key in XGB_BAYES_KEYS}
    esn_params["res_size"] = int(esn_params["res_size"])
    esn_params["washout"] = int(esn_params["washout"])
    xgb_params["n_estimators"] = int(xgb_params["n_estimators"])
    xgb_params["max_depth"] = int(xgb_params["max_depth"])
    return esn_params, xgb_params


def _predict_joint_fold(train_meta, valid_meta, target, params, random_states):
    train_ids = train_meta["trial_id"].tolist()
    valid_ids = valid_meta["trial_id"].tolist()
    if set(train_ids) & set(valid_ids):
        raise RuntimeError("Trial leakage detected during Bayesian search.")
    if set(train_meta["Sample"]) & set(valid_meta["Sample"]):
        raise RuntimeError("Material leakage detected during Bayesian search.")

    esn_params, xgb_params = _split_joint_parameters(params)
    seed_predictions = []
    for random_state in random_states:
        features = build_esn_feature_table(
            train_ids,
            train_ids + valid_ids,
            esn_params=esn_params,
            random_state=random_state,
        ).set_index("trial_id")
        feature_cols = ["Temperature_C"] + [
            column for column in features if column.startswith("esn_")
        ]
        X_train = features.loc[train_ids, feature_cols].reset_index(drop=True)
        X_valid = features.loc[valid_ids, feature_cols].reset_index(drop=True)
        y_train = features.loc[train_ids, target].reset_index(drop=True)

        model = make_regressor(xgb_params=xgb_params, random_state=random_state)
        model.fit(X_train, y_train)
        seed_predictions.append(np.clip(
            model.predict(X_valid), y_train.min(), y_train.max()
        ))

    return valid_meta[target].to_numpy(float), np.mean(seed_predictions, axis=0)


def _inner_objective(metadata, target, params, n_splits, random_states, stability_weight):
    splitter = GroupKFold(n_splits=n_splits)
    rows = []
    for fold, (train_idx, valid_idx) in enumerate(
        splitter.split(metadata, groups=metadata["Sample"]), start=1
    ):
        y_true, y_pred = _predict_joint_fold(
            metadata.iloc[train_idx], metadata.iloc[valid_idx],
            target, params, random_states,
        )
        rows.append({"fold": fold, **regression_metrics(y_true, y_pred)})

    fold_metrics = pd.DataFrame(rows)
    values = fold_metrics["nrmse_range"]
    if len(values) != n_splits or np.any(~np.isfinite(values)):
        raise ValueError("Inner-fold NRMSE contains invalid values.")
    mean_nrmse = float(values.mean())
    sd_nrmse = float(values.std(ddof=0))
    return mean_nrmse + stability_weight * sd_nrmse, fold_metrics


def bayesian_optimize_joint(
    metadata,
    target="eff",
    n_trials=BAYES_N_TRIALS,
    n_initial=BAYES_N_INITIAL,
    inner_splits=4,
    random_states=(42, 43, 44),
    stability_weight=BAYES_STABILITY_WEIGHT,
    acquisition_candidates=BAYES_ACQUISITION_CANDIDATES,
    optimizer_seed=2026,
):
    """Minimize grouped-CV loss with GP expected improvement over joint parameters."""
    if n_trials < 2 or not 1 <= n_initial <= n_trials:
        raise ValueError("Require n_trials >= 2 and 1 <= n_initial <= n_trials.")
    if inner_splits < 2 or inner_splits > metadata["Sample"].nunique():
        raise ValueError("Invalid number of inner material folds.")
    if not random_states:
        raise ValueError("At least one reservoir seed is required.")

    rng = np.random.default_rng(optimizer_seed)
    points, losses, rows = [], [], []

    for trial_number in range(1, n_trials + 1):
        if trial_number <= n_initial:
            point = rng.random(len(BAYES_KEYS))
            acquisition = "random_initialization"
        else:
            X_observed = np.asarray(points)
            y_observed = np.asarray(losses)
            kernel = (
                ConstantKernel(1.0, (1e-2, 1e2))
                * Matern(length_scale=np.ones(len(BAYES_KEYS)), nu=2.5)
                + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e-1))
            )
            gp = GaussianProcessRegressor(
                kernel=kernel,
                normalize_y=True,
                random_state=optimizer_seed + trial_number,
                n_restarts_optimizer=2,
            )
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                gp.fit(X_observed, y_observed)
            candidates = rng.random((acquisition_candidates, len(BAYES_KEYS)))
            mean, std = gp.predict(candidates, return_std=True)
            improvement = np.min(y_observed) - mean - 0.01
            z = np.divide(improvement, std, out=np.zeros_like(std), where=std > 0)
            expected_improvement = improvement * norm.cdf(z) + std * norm.pdf(z)
            point = candidates[int(np.argmax(expected_improvement))]
            acquisition = "expected_improvement"

        params = _decode_bayes_point(point)
        loss, fold_metrics = _inner_objective(
            metadata, target, params, inner_splits,
            tuple(random_states), stability_weight,
        )
        points.append(point)
        losses.append(loss)
        rows.append({
            "trial": trial_number,
            "acquisition": acquisition,
            **params,
            "mean_inner_nrmse": float(fold_metrics["nrmse_range"].mean()),
            "sd_inner_nrmse": float(fold_metrics["nrmse_range"].std(ddof=0)),
            "mean_inner_r2": float(fold_metrics["r2"].mean()),
            "performance_index": loss,
        })
        print(f"  Joint Bayesian trial {trial_number:02d}/{n_trials}: J={loss:.4f}")

    history = pd.DataFrame(rows).sort_values("performance_index").reset_index(drop=True)
    best_params = {key: history.loc[0, key] for key in BAYES_KEYS}
    best_params["res_size"] = int(best_params["res_size"])
    best_params["washout"] = int(best_params["washout"])
    best_params["n_estimators"] = int(best_params["n_estimators"])
    best_params["max_depth"] = int(best_params["max_depth"])
    return history, best_params


def nested_lomo_joint_bayesian_search(
    target="eff",
    n_trials=BAYES_N_TRIALS,
    n_initial=BAYES_N_INITIAL,
    inner_splits=4,
    random_states=(42, 43, 44),
    stability_weight=BAYES_STABILITY_WEIGHT,
):
    metadata = THERMAL_FEATURES[
        ["trial_id", "Temperature", "Temperature_C", "Sample", "Trial", "k", "eff"]
    ].reset_index(drop=True)
    outer_splitter = LeaveOneGroupOut()
    search_rows, outer_rows, prediction_rows = [], [], []

    splits = list(outer_splitter.split(metadata, groups=metadata["Sample"]))
    for outer_fold, (train_idx, test_idx) in enumerate(splits, start=1):
        outer_train = metadata.iloc[train_idx].reset_index(drop=True)
        outer_test = metadata.iloc[test_idx].reset_index(drop=True)
        held_out = outer_test["Sample"].iloc[0]
        print(f"Outer fold {outer_fold}/{len(splits)}: held out {held_out}")

        history, best_params = bayesian_optimize_joint(
            outer_train,
            target=target,
            n_trials=n_trials,
            n_initial=n_initial,
            inner_splits=inner_splits,
            random_states=random_states,
            stability_weight=stability_weight,
            optimizer_seed=2026 + outer_fold,
        )
        history.insert(0, "outer_fold", outer_fold)
        history.insert(1, "held_out_material", held_out)
        search_rows.append(history)

        y_true, y_pred = _predict_joint_fold(
            outer_train, outer_test, target, best_params, tuple(random_states)
        )
        outer_diagnostics = constant_target_fold_metrics(
            y_true, y_pred, np.ptp(outer_train[target].to_numpy(float))
        )
        outer_rows.append({
            "outer_fold": outer_fold,
            "held_out_material": held_out,
            **best_params,
            **outer_diagnostics,
        })
        prediction_rows.append(pd.DataFrame({
            "trial_id": outer_test["trial_id"].to_numpy(),
            "Temperature": outer_test["Temperature"].to_numpy(),
            "Temperature_C": outer_test["Temperature_C"].to_numpy(),
            "Sample": outer_test["Sample"].to_numpy(),
            "Trial": outer_test["Trial"].to_numpy(),
            "outer_fold": outer_fold,
            "y_true": y_true,
            "y_pred": y_pred,
        }))

    searches = pd.concat(search_rows, ignore_index=True)
    outer_results = pd.DataFrame(outer_rows)
    predictions = pd.concat(prediction_rows, ignore_index=True)
    nested_metrics = pd.DataFrame([{
        "target": target,
        "protocol": "multi_temp_nested_lomo_joint_bayesian_esn_xgboost",
        "n_bayesian_trials": n_trials,
        "random_states": tuple(random_states),
        **regression_metrics(predictions["y_true"], predictions["y_pred"]),
    }])

    print("Final joint Bayesian search on all materials for deployment parameters")
    final_history, recommended = bayesian_optimize_joint(
        metadata,
        target=target,
        n_trials=n_trials,
        n_initial=n_initial,
        inner_splits=inner_splits,
        random_states=random_states,
        stability_weight=stability_weight,
        optimizer_seed=4042,
    )
    return searches, outer_results, predictions, nested_metrics, final_history, recommended


In [11]:
BAYES_TARGET = OPTIMIZATION_TARGET
RANDOM_STATES = RESERVOIR_SEEDS

(
    BAYES_SEARCH_HISTORY,
    BAYES_OUTER_RESULTS,
    BAYES_PREDICTIONS,
    BAYES_NESTED_METRICS,
    BAYES_FINAL_HISTORY,
    RECOMMENDED_JOINT_PARAMETERS,
) = nested_lomo_joint_bayesian_search(
    target=BAYES_TARGET,
    n_trials=BAYES_N_TRIALS,
    n_initial=BAYES_N_INITIAL,
    inner_splits=INNER_MATERIAL_FOLDS,
    random_states=RANDOM_STATES,
    stability_weight=BAYES_STABILITY_WEIGHT,
)

RECOMMENDED_ESN_PARAMETERS, RECOMMENDED_XGB_PARAMETERS = _split_joint_parameters(
    RECOMMENDED_JOINT_PARAMETERS
)

BAYES_OUTER_STABILITY = pd.DataFrame([{
    "folds": len(BAYES_OUTER_RESULTS),
    "mean_fold_mae": BAYES_OUTER_RESULTS["mae"].mean(),
    "sd_fold_mae": BAYES_OUTER_RESULTS["mae"].std(ddof=1),
    "worst_fold_mae": BAYES_OUTER_RESULTS["mae"].max(),
    "mean_fold_rmse": BAYES_OUTER_RESULTS["rmse"].mean(),
    "sd_fold_rmse": BAYES_OUTER_RESULTS["rmse"].std(ddof=1),
    "worst_fold_rmse": BAYES_OUTER_RESULTS["rmse"].max(),
    "mean_fold_nrmse_training_range": BAYES_OUTER_RESULTS["nrmse_training_range"].mean(),
    "sd_fold_nrmse_training_range": BAYES_OUTER_RESULTS["nrmse_training_range"].std(ddof=1),
    "worst_fold_nrmse_training_range": BAYES_OUTER_RESULTS["nrmse_training_range"].max(),
}])

temperature_rows = []
for temperature, part in BAYES_PREDICTIONS.groupby("Temperature", sort=True):
    temperature_rows.append({
        "Temperature": temperature,
        **regression_metrics(part["y_true"], part["y_pred"]),
    })
TEMPERATURE_OOF_METRICS = pd.DataFrame(temperature_rows)

print("Unbiased nested multi-temperature LOMO performance:")
display(BAYES_NESTED_METRICS)
print("Per-material outer-fold diagnostics:")
display(BAYES_OUTER_RESULTS)
print("Across-material stability:")
display(BAYES_OUTER_STABILITY)
print("Nested OOF performance by operating temperature:")
display(TEMPERATURE_OOF_METRICS)
print("Final recommended ESN parameters:")
display(pd.Series(RECOMMENDED_ESN_PARAMETERS))
print("Final recommended XGBoost parameters:")
display(pd.Series(RECOMMENDED_XGB_PARAMETERS))

BAYES_SEARCH_HISTORY.to_csv(RESULTS_DIR / f"{RESULT_STEM}_joint_search_history.csv", index=False)
BAYES_OUTER_RESULTS.to_csv(RESULTS_DIR / f"{RESULT_STEM}_outer_fold_metrics.csv", index=False)
BAYES_PREDICTIONS.to_csv(RESULTS_DIR / f"{RESULT_STEM}_nested_oof_predictions.csv", index=False)
BAYES_NESTED_METRICS.to_csv(RESULTS_DIR / f"{RESULT_STEM}_pooled_oof_metrics.csv", index=False)
BAYES_OUTER_STABILITY.to_csv(RESULTS_DIR / f"{RESULT_STEM}_fold_stability.csv", index=False)
TEMPERATURE_OOF_METRICS.to_csv(RESULTS_DIR / f"{RESULT_STEM}_temperature_oof_metrics.csv", index=False)
BAYES_FINAL_HISTORY.to_csv(RESULTS_DIR / f"{RESULT_STEM}_final_joint_search_history.csv", index=False)

payload = {
    "source": "final all-material multi-temperature joint Bayesian search after nested LOMO",
    "temperature_folders": list(TEMPERATURE_FOLDERS),
    "target": BAYES_TARGET,
    "analysis_window": [float(value) for value in ANALYSIS_WINDOW],
    "thermal_windows": {
        "early": list(EARLY_WINDOW), "mid": list(MID_WINDOW), "late": list(LATE_WINDOW),
    },
    "summary_features": [
        "mean", "std", "range", "net_change", "slope", "absolute_area",
        "time_of_max_absolute", "skewness", "excess_kurtosis",
    ],
    "reservoir_seeds": [int(value) for value in RANDOM_STATES],
    "esn_parameters": {
        key: (int(value) if key in {"res_size", "washout"} else float(value))
        for key, value in RECOMMENDED_ESN_PARAMETERS.items()
    },
    "xgb_parameters": {
        key: (int(value) if key in {"n_estimators", "max_depth"} else float(value))
        for key, value in RECOMMENDED_XGB_PARAMETERS.items()
    },
    "bayesian_search": {
        "type": "joint_esn_xgboost_gaussian_process_expected_improvement",
        "n_trials": int(BAYES_N_TRIALS), "n_initial": int(BAYES_N_INITIAL),
        "inner_material_folds": int(INNER_MATERIAL_FOLDS),
        "stability_weight": float(BAYES_STABILITY_WEIGHT),
        "search_space": {
            key: [kind, float(low), float(high)]
            for key, (kind, low, high) in BAYES_SPACE.items()
        },
    },
}
with PARAMETER_FILE.open("w") as file:
    json.dump(payload, file, indent=2)
print("Saved configuration for 4B:", PARAMETER_FILE)


Outer fold 1/14: held out aluminum
  Joint Bayesian trial 01/50: J=3.7324
  Joint Bayesian trial 02/50: J=3.1894
  Joint Bayesian trial 03/50: J=3.2694
  Joint Bayesian trial 04/50: J=2.6621
  Joint Bayesian trial 05/50: J=2.4751
  Joint Bayesian trial 06/50: J=3.5333
  Joint Bayesian trial 07/50: J=4.1500
  Joint Bayesian trial 08/50: J=3.8768
  Joint Bayesian trial 09/50: J=3.6730
  Joint Bayesian trial 10/50: J=3.5246
  Joint Bayesian trial 11/50: J=2.6555
  Joint Bayesian trial 12/50: J=3.6707
  Joint Bayesian trial 13/50: J=3.9488
  Joint Bayesian trial 14/50: J=3.3593
  Joint Bayesian trial 15/50: J=3.5395
  Joint Bayesian trial 16/50: J=2.7015
  Joint Bayesian trial 17/50: J=3.1085
  Joint Bayesian trial 18/50: J=2.4098
  Joint Bayesian trial 19/50: J=2.5866
  Joint Bayesian trial 20/50: J=3.0603
  Joint Bayesian trial 21/50: J=3.0151
  Joint Bayesian trial 22/50: J=3.5462
  Joint Bayesian trial 23/50: J=3.7270
  Joint Bayesian trial 24/50: J=3.2002
  Joint Bayesian trial 25/50:

,target,protocol,n_bayesian_trials,random_states,n,mae,rmse,nrmse_range,r2,median_ape_pct
0,eff,multi_temp_nested_lomo_joint_bayesian_esn_xgboost,50,"(42, 43, 44)",336,3638.445129,7278.245735,0.195872,0.578321,44.051119


Per-material outer-fold diagnostics:


,outer_fold,held_out_material,res_size,leak_rate,input_magnitude,spectral_radius,washout,n_estimators,max_depth,learning_rate,...,subsample,colsample_bytree,reg_alpha,reg_lambda,n,mae,rmse,nrmse_training_range,mean_error_bias,median_ape_pct
0,1,aluminum,22,0.750700,1.763002,1.186496,1,536,3,0.035526,...,0.896022,0.384485,2.455210,35.045065,24,7407.464844,7988.012305,0.214974,-7407.464844,31.458855
1,2,bismuth,35,0.847685,1.255663,1.413345,1,104,3,0.011887,...,0.712154,0.317998,0.078762,20.324527,24,2543.706035,2932.777914,0.078927,2451.799881,86.765224
2,3,cement,72,0.314354,0.626806,1.042665,3,594,2,0.099222,...,0.732825,0.669221,0.010462,27.555906,24,1221.865833,2324.293495,0.062551,1071.109198,38.896978
3,4,copper,53,0.147621,0.556372,1.212341,0,584,2,0.139567,...,0.721568,0.651127,0.005464,88.879148,24,23604.192093,24072.124747,1.006208,-23604.192093,59.892613
4,5,cork,48,0.241586,1.814865,1.489577,2,569,3,0.135641,...,0.918129,0.633254,0.000194,11.493930,24,133.187155,162.842867,0.004382,92.134629,75.584939
5,6,graphite,42,0.533305,1.876649,1.105898,2,369,3,0.132218,...,0.956399,0.732185,0.152159,0.140206,24,3295.258288,4162.763368,0.112028,-744.998758,21.600595
6,7,gypsum,41,0.695067,1.948964,1.397071,4,209,2,0.064243,...,0.818260,0.547000,0.039481,0.121915,24,1079.482487,1624.618188,0.043722,979.516985,141.135522
7,8,iron,32,0.466490,0.825291,1.401983,1,598,2,0.028943,...,0.631197,0.945792,4.222611,3.871979,24,3662.141052,4596.216958,0.123694,1948.208515,21.879742
8,9,nickel,79,0.213780,0.719755,0.876406,3,514,2,0.011292,...,0.799903,0.991609,0.307322,0.292781,24,5507.210926,6193.826376,0.166688,-5489.101246,29.192674
9,10,pdms,51,0.898765,1.071853,0.500856,1,234,2,0.079576,...,0.726157,0.391375,0.058534,9.499984,24,196.193720,226.651459,0.006100,-192.074754,42.185648


Across-material stability:


...

Final recommended ESN parameters:


res_size           31.000000
leak_rate           0.345891
input_magnitude     0.733466
spectral_radius     1.224867
washout             4.000000
dtype: float64

Final recommended XGBoost parameters:


n_estimators        103.000000
max_depth             3.000000
learning_rate         0.010463
min_child_weight      8.493270
subsample             0.728033
colsample_bytree      0.979219
reg_alpha             0.003630
reg_lambda           69.856590
dtype: float64

Saved configuration for 4B: /Users/markkeanujamesexconde/Library/CloudStorage/OneDrive-Personal/Documents/Academe Files/4. Doctoral/Research 1 - Warmth Sensor/Warmth-Sensor-Object-Detection/results/multi_temp_esn/30C-40C-50C-60C_eff_v5_final_parameters.json


## Reading 4A outputs

- The pooled nested outer OOF metrics are the primary unseen-material estimate.
- Temperature-specific OOF metrics reveal whether performance depends on operating condition.
- Outer-fold parameter combinations are diagnostic and must not be averaged.
- The JSON file contains the final all-material configuration consumed by 4B.


# 9. Handoff

Run 4A when the pooled datasets, windows, summaries, or search design change. Use 4B
for routine fixed-parameter LORO, LOMO, LOTO, tables, and plots.
